# Chapter 6 Lab — Tool Use

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 6 — Tool Use  
**Goal:** Understand how LLMs call external tools, build a tool registry and dispatcher from scratch, and learn patterns for composition, error handling, and security.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | The Hallucination Problem |
| 3 | The Function Calling Protocol |
| 4 | Tool Definitions: JSON Schema |
| 5 | Tool Registry Architecture |
| 6 | The Dispatcher |
| 7 | Result Parsing |
| 8 | Dynamic Tool Selection |
| 9 | Tool Composition |
| 10 | Error Handling Patterns |
| 11 | Security Considerations |
| 12 | Exercises |

> **Prerequisites:** Python 3.10+. An OpenAI API key in a `.env` file is optional — all examples include mock responses so the notebook runs without one.

---
## 1. Setup

In [ ]:
# Run once to install required packages (restart the kernel afterwards).

%pip install openai python-dotenv --quiet

In [ ]:
import json
import os
import re
import textwrap
import time
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Callable

from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env if present

# ── Optional: live API client ────────────────────────────────────────────────
# Set USE_LIVE_API = True if you have a valid key and want real LLM responses.
USE_LIVE_API = False

if USE_LIVE_API:
    import openai
    client = openai.OpenAI()
    print("Live API mode enabled.")
else:
    client = None
    print("Mock mode — no API key required. All LLM responses are simulated.")

---
## 2. The Hallucination Problem

LLMs generate plausible text, but they have **no access to real-time data** and can confidently invent facts. Tools solve this by giving the model a way to *look things up* instead of guessing.

In [ ]:
# ── Simulated: an LLM answering without tools ────────────────────────────────

HALLUCINATED_RESPONSE = {
    "question": "What is the current price of AAPL stock?",
    "llm_answer": "As of today, Apple (AAPL) is trading at $187.42 per share.",
    "problem": (
        "The model invented a price. It has no market data feed. "
        "The number looks real, which makes it dangerous."
    ),
}

print(f"Q: {HALLUCINATED_RESPONSE['question']}")
print(f"A: {HALLUCINATED_RESPONSE['llm_answer']}")
print(f"\nProblem: {HALLUCINATED_RESPONSE['problem']}")

In [ ]:
# ── The fix: ground the answer with a tool call ──────────────────────────────

def mock_stock_lookup(symbol: str) -> dict:
    """Simulates an API call to a stock price service."""
    prices = {"AAPL": 214.29, "GOOGL": 178.36, "MSFT": 441.05}
    if symbol in prices:
        return {"symbol": symbol, "price": prices[symbol], "currency": "USD"}
    return {"error": f"Unknown symbol: {symbol}"}


# Tool provides verified data; the LLM formats it for the user.
tool_result = mock_stock_lookup("AAPL")
grounded_answer = f"AAPL is currently trading at ${tool_result['price']:.2f} USD."

print(f"Tool result : {tool_result}")
print(f"Grounded answer: {grounded_answer}")
print("\nThe model no longer needs to guess — it reads the tool output.")

**Key insight:** Tools don't make the LLM smarter. They give it **access to ground truth** so it can stop hallucinating on questions that require external data.

---
## 3. The Function Calling Protocol

OpenAI's function calling works in three phases:

1. **You** send the model a list of available tools (JSON schemas) alongside the user message.
2. **The model** responds with a `tool_calls` array instead of (or alongside) a text reply.
3. **You** execute the functions locally and send results back as `tool` role messages.

Let's walk through each phase manually.

In [ ]:
# ── Phase 1: Define a tool for the model ─────────────────────────────────────

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'San Francisco'",
                },
                "units": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature units",
                },
            },
            "required": ["city"],
        },
    },
}

print(json.dumps(weather_tool, indent=2))

In [ ]:
# ── Phase 2: Model responds with a tool_call (simulated) ─────────────────────

# In a real call, this is what the API returns inside
# response.choices[0].message.tool_calls[0]

mock_tool_call = {
    "id": "call_abc123",
    "type": "function",
    "function": {
        "name": "get_weather",
        "arguments": json.dumps({"city": "San Francisco", "units": "celsius"}),
    },
}

print("Model chose to call:")
print(f"  Function : {mock_tool_call['function']['name']}")
print(f"  Arguments: {mock_tool_call['function']['arguments']}")
print(f"  Call ID  : {mock_tool_call['id']}")

In [ ]:
# ── Phase 3: Execute locally, send result back ───────────────────────────────

def get_weather(city: str, units: str = "celsius") -> str:
    """Mock weather lookup."""
    data = {
        "San Francisco": {"temp_c": 16, "condition": "Foggy"},
        "Tokyo": {"temp_c": 22, "condition": "Clear"},
        "London": {"temp_c": 12, "condition": "Rainy"},
    }
    info = data.get(city, {"temp_c": 20, "condition": "Unknown"})
    temp = info["temp_c"] if units == "celsius" else round(info["temp_c"] * 9 / 5 + 32)
    unit_label = "C" if units == "celsius" else "F"
    return json.dumps({"city": city, "temperature": f"{temp}{unit_label}", "condition": info["condition"]})


# Execute the tool
args = json.loads(mock_tool_call["function"]["arguments"])
result = get_weather(**args)

# This is what you send back to the API as a 'tool' message
tool_message = {
    "role": "tool",
    "tool_call_id": mock_tool_call["id"],
    "content": result,
}

print("Tool result sent back to model:")
print(json.dumps(tool_message, indent=2))

In [ ]:
# ── Full conversation flow (mock) ────────────────────────────────────────────
# Showing the complete message sequence the API expects.

full_conversation = [
    # 1. User asks a question
    {"role": "user", "content": "What's the weather in San Francisco?"},
    # 2. Assistant responds with a tool call (no text content)
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [mock_tool_call],
    },
    # 3. Tool result
    tool_message,
    # 4. Assistant gives final answer using the tool result
    {
        "role": "assistant",
        "content": "It's currently 16C and foggy in San Francisco.",
    },
]

for msg in full_conversation:
    role = msg["role"].upper()
    if msg.get("tool_calls"):
        print(f"[{role}] -> tool_call: {msg['tool_calls'][0]['function']['name']}({msg['tool_calls'][0]['function']['arguments']})")
    elif msg.get("tool_call_id"):
        print(f"[{role}] -> {msg['content']}")
    else:
        print(f"[{role}] {msg.get('content', '')}")

---
## 4. Tool Definitions: JSON Schema

Every tool the model can call needs a **JSON Schema** describing its parameters. Writing these by hand is tedious and error-prone, so let's build a helper that generates schemas from Python type hints.

In [ ]:
import inspect
from typing import get_type_hints


# Mapping Python types to JSON Schema types
PY_TO_JSON_TYPE = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}


def schema_from_function(func: Callable, description: str = "") -> dict:
    """Generate an OpenAI tool schema from a Python function's signature."""
    hints = get_type_hints(func)
    sig = inspect.signature(func)
    
    properties = {}
    required = []
    
    for name, param in sig.parameters.items():
        py_type = hints.get(name, str)
        json_type = PY_TO_JSON_TYPE.get(py_type, "string")
        properties[name] = {"type": json_type, "description": f"Parameter: {name}"}
        
        # If no default value, it's required
        if param.default is inspect.Parameter.empty:
            required.append(name)
    
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": description or (func.__doc__ or "").strip(),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


# ── Test it ───────────────────────────────────────────────────────────────────

def calculate(expression: str, precision: int = 2) -> float:
    """Evaluate a mathematical expression and return the result."""
    pass  # implementation later


auto_schema = schema_from_function(calculate)
print(json.dumps(auto_schema, indent=2))

In [ ]:
# ── Enhance with custom descriptions per parameter ───────────────────────────

def schema_from_function_v2(
    func: Callable,
    description: str = "",
    param_descriptions: dict[str, str] | None = None,
    enum_values: dict[str, list] | None = None,
) -> dict:
    """Enhanced schema generator with per-parameter descriptions and enums."""
    hints = get_type_hints(func)
    sig = inspect.signature(func)
    param_descriptions = param_descriptions or {}
    enum_values = enum_values or {}

    properties = {}
    required = []

    for name, param in sig.parameters.items():
        py_type = hints.get(name, str)
        json_type = PY_TO_JSON_TYPE.get(py_type, "string")
        prop: dict[str, Any] = {
            "type": json_type,
            "description": param_descriptions.get(name, f"Parameter: {name}"),
        }
        if name in enum_values:
            prop["enum"] = enum_values[name]
        properties[name] = prop

        if param.default is inspect.Parameter.empty:
            required.append(name)

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": description or (func.__doc__ or "").strip(),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


rich_schema = schema_from_function_v2(
    get_weather,
    description="Get the current weather for a city.",
    param_descriptions={
        "city": "City name, e.g. 'San Francisco'",
        "units": "Temperature units to use",
    },
    enum_values={"units": ["celsius", "fahrenheit"]},
)
print(json.dumps(rich_schema, indent=2))

---
## 5. Tool Registry Architecture

A **Tool Registry** is the central catalog of every tool your agent can use. It stores the function, its schema, and metadata. This is the foundation for dispatching, dynamic selection, and composition.

In [ ]:
@dataclass
class ToolSpec:
    """Metadata and implementation for a single tool."""
    name: str
    description: str
    func: Callable
    schema: dict
    tags: list[str] = field(default_factory=list)
    version: str = "1.0"


class ToolRegistry:
    """Central catalog of agent tools."""

    def __init__(self):
        self._tools: dict[str, ToolSpec] = {}

    # ── Core API ──────────────────────────────────────────────────────────────

    def register(
        self,
        func: Callable,
        *,
        name: str | None = None,
        description: str = "",
        param_descriptions: dict[str, str] | None = None,
        enum_values: dict[str, list] | None = None,
        tags: list[str] | None = None,
    ) -> "ToolRegistry":
        """Register a function as a tool. Returns self for chaining."""
        tool_name = name or func.__name__
        schema = schema_from_function_v2(
            func,
            description=description,
            param_descriptions=param_descriptions,
            enum_values=enum_values,
        )
        # Override the name in the schema if a custom name was given
        schema["function"]["name"] = tool_name

        self._tools[tool_name] = ToolSpec(
            name=tool_name,
            description=description or (func.__doc__ or "").strip(),
            func=func,
            schema=schema,
            tags=tags or [],
        )
        return self

    def get(self, name: str) -> ToolSpec | None:
        """Look up a tool by name."""
        return self._tools.get(name)

    def list_tools(self) -> list[str]:
        """Return all registered tool names."""
        return list(self._tools.keys())

    def list_by_tag(self, tag: str) -> list[str]:
        """Return tool names that have a specific tag."""
        return [name for name, spec in self._tools.items() if tag in spec.tags]

    def schemas(self, names: list[str] | None = None) -> list[dict]:
        """Return OpenAI-compatible tool schemas, optionally filtered by name."""
        if names is None:
            return [spec.schema for spec in self._tools.values()]
        return [self._tools[n].schema for n in names if n in self._tools]

    def __len__(self) -> int:
        return len(self._tools)

    def __repr__(self) -> str:
        return f"ToolRegistry({self.list_tools()})"


print("ToolRegistry class defined.")

In [ ]:
# ── Build a registry and register tools ──────────────────────────────────────

def calculate_expr(expression: str) -> str:
    """Safely evaluate a math expression."""
    allowed = set("0123456789+-*/.() ")
    if not all(ch in allowed for ch in expression):
        return json.dumps({"error": "Invalid characters in expression"})
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})


def lookup_word(word: str) -> str:
    """Look up the definition of a word."""
    definitions = {
        "agentic": "Relating to or denoting AI systems that can take autonomous actions.",
        "hallucination": "When an AI model generates factually incorrect information presented as truth.",
        "grounding": "Connecting AI outputs to verified external data sources.",
    }
    defn = definitions.get(word.lower(), f"No definition found for '{word}'.")
    return json.dumps({"word": word, "definition": defn})


def get_current_time() -> str:
    """Return the current date and time."""
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    return json.dumps({"utc": now.isoformat(), "unix": int(now.timestamp())})


# ── Register ──────────────────────────────────────────────────────────────────

registry = ToolRegistry()

registry.register(
    get_weather,
    description="Get the current weather for a city.",
    param_descriptions={"city": "City name", "units": "Temperature units"},
    enum_values={"units": ["celsius", "fahrenheit"]},
    tags=["data", "weather"],
)

registry.register(
    calculate_expr,
    name="calculator",
    description="Evaluate a mathematical expression.",
    param_descriptions={"expression": "A math expression like '2 + 3 * 4'"},
    tags=["math", "utility"],
)

registry.register(
    lookup_word,
    description="Look up the definition of a word.",
    param_descriptions={"word": "The word to define"},
    tags=["data", "reference"],
)

registry.register(
    get_current_time,
    description="Return the current UTC date and time.",
    tags=["utility"],
)

print(f"Registered {len(registry)} tools: {registry.list_tools()}")
print(f"Tools tagged 'data': {registry.list_by_tag('data')}")
print(f"Tools tagged 'utility': {registry.list_by_tag('utility')}")

In [ ]:
# ── Inspect a registered tool's schema ───────────────────────────────────────

spec = registry.get("calculator")
print(f"Name   : {spec.name}")
print(f"Tags   : {spec.tags}")
print(f"Version: {spec.version}")
print(f"Schema :")
print(json.dumps(spec.schema, indent=2))

---
## 6. The Dispatcher

The **dispatcher** sits between the LLM and your tool implementations. It receives the model's `tool_calls`, looks up each function in the registry, executes it, and returns formatted results.

```
LLM  -->  tool_calls  -->  Dispatcher  -->  Registry  -->  Function
                                  |                            |
                                  <----  tool messages  <------
```

In [ ]:
@dataclass
class ToolCall:
    """Represents a single tool call from the model."""
    id: str
    name: str
    arguments: dict


@dataclass
class ToolResult:
    """Result of executing a tool call."""
    call_id: str
    name: str
    content: str
    success: bool
    elapsed_ms: float = 0.0


class Dispatcher:
    """Routes tool calls to registry implementations and collects results."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def parse_tool_calls(self, raw_tool_calls: list[dict]) -> list[ToolCall]:
        """Parse raw OpenAI tool_calls into ToolCall objects."""
        calls = []
        for tc in raw_tool_calls:
            args_str = tc["function"]["arguments"]
            args = json.loads(args_str) if isinstance(args_str, str) else args_str
            calls.append(ToolCall(id=tc["id"], name=tc["function"]["name"], arguments=args))
        return calls

    def execute(self, call: ToolCall) -> ToolResult:
        """Execute a single tool call."""
        start = time.perf_counter()
        spec = self.registry.get(call.name)

        if spec is None:
            return ToolResult(
                call_id=call.id,
                name=call.name,
                content=json.dumps({"error": f"Unknown tool: {call.name}"}),
                success=False,
            )

        try:
            result = spec.func(**call.arguments)
            elapsed = (time.perf_counter() - start) * 1000
            return ToolResult(
                call_id=call.id, name=call.name,
                content=result, success=True, elapsed_ms=round(elapsed, 2),
            )
        except Exception as e:
            elapsed = (time.perf_counter() - start) * 1000
            return ToolResult(
                call_id=call.id, name=call.name,
                content=json.dumps({"error": str(e)}),
                success=False, elapsed_ms=round(elapsed, 2),
            )

    def dispatch(self, raw_tool_calls: list[dict]) -> list[ToolResult]:
        """Parse and execute all tool calls, returning results."""
        calls = self.parse_tool_calls(raw_tool_calls)
        return [self.execute(call) for call in calls]


dispatcher = Dispatcher(registry)
print("Dispatcher ready.")

In [ ]:
# ── Test the dispatcher with mock tool calls ─────────────────────────────────

mock_calls = [
    {
        "id": "call_001",
        "type": "function",
        "function": {"name": "get_weather", "arguments": json.dumps({"city": "Tokyo"})},
    },
    {
        "id": "call_002",
        "type": "function",
        "function": {"name": "calculator", "arguments": json.dumps({"expression": "42 * 1.15"})},
    },
    {
        "id": "call_003",
        "type": "function",
        "function": {"name": "nonexistent_tool", "arguments": "{}"},
    },
]

results = dispatcher.dispatch(mock_calls)

for r in results:
    status = "OK" if r.success else "FAIL"
    print(f"[{status}] {r.name} ({r.elapsed_ms}ms) -> {r.content}")

---
## 7. Result Parsing

Tool outputs need to be formatted as messages the LLM can consume. The model expects each result as a `{"role": "tool", "tool_call_id": ..., "content": ...}` message.

In [ ]:
def results_to_messages(results: list[ToolResult]) -> list[dict]:
    """Convert ToolResult objects into OpenAI tool messages."""
    messages = []
    for r in results:
        messages.append({
            "role": "tool",
            "tool_call_id": r.call_id,
            "content": r.content,
        })
    return messages


tool_messages = results_to_messages(results)
for msg in tool_messages:
    print(json.dumps(msg, indent=2))
    print()

In [ ]:
# ── Smart result formatter: truncate, summarize, add metadata ────────────────

def format_result(result: ToolResult, max_length: int = 2000) -> dict:
    """Format a tool result with truncation and metadata annotation."""
    content = result.content

    # Truncate very long results so we don't blow up the context window
    if len(content) > max_length:
        content = content[:max_length] + f"\n... [TRUNCATED — {len(result.content)} chars total]"

    # Add metadata as a prefix so the model knows what happened
    meta = f"[Tool: {result.name} | Success: {result.success} | {result.elapsed_ms}ms]\n"

    return {
        "role": "tool",
        "tool_call_id": result.call_id,
        "content": meta + content,
    }


# Test with a successful result
formatted = format_result(results[0])
print(formatted["content"])
print()

# Test with a failed result
formatted_err = format_result(results[2])
print(formatted_err["content"])

---
## 8. Dynamic Tool Selection

Sending **all** tools to the model on every call wastes tokens and can confuse the model. Dynamic tool selection filters the tool list based on the user's query, tags, or embeddings.

In [ ]:
# ── Strategy 1: Keyword-based selection ──────────────────────────────────────

KEYWORD_MAP = {
    "weather": ["get_weather"],
    "temperature": ["get_weather"],
    "forecast": ["get_weather"],
    "calculate": ["calculator"],
    "math": ["calculator"],
    "compute": ["calculator"],
    "define": ["lookup_word"],
    "definition": ["lookup_word"],
    "meaning": ["lookup_word"],
    "time": ["get_current_time"],
    "date": ["get_current_time"],
}


def select_tools_by_keyword(query: str, registry: ToolRegistry) -> list[str]:
    """Select relevant tools based on keywords in the query."""
    query_lower = query.lower()
    selected = set()
    for keyword, tool_names in KEYWORD_MAP.items():
        if keyword in query_lower:
            selected.update(tool_names)

    # Always include at least some tools as fallback
    if not selected:
        return registry.list_tools()
    return list(selected)


# Test
queries = [
    "What's the weather in Paris?",
    "Calculate 15% tip on $84.50",
    "What does 'agentic' mean?",
    "Tell me a joke",  # no match -> returns all tools
]

for q in queries:
    tools = select_tools_by_keyword(q, registry)
    print(f"Q: {q}")
    print(f"   Selected: {tools}\n")

In [ ]:
# ── Strategy 2: Tag-based selection ──────────────────────────────────────────

def select_tools_by_tag(tags: list[str], registry: ToolRegistry) -> list[str]:
    """Select tools that match any of the given tags."""
    selected = set()
    for tag in tags:
        selected.update(registry.list_by_tag(tag))
    return list(selected)


print("Data tools: ", select_tools_by_tag(["data"], registry))
print("Utility tools:", select_tools_by_tag(["utility"], registry))
print("Math + data: ", select_tools_by_tag(["math", "data"], registry))

In [ ]:
# ── Strategy 3: Embedding-based selection (conceptual) ────────────────────────
# In production you would embed tool descriptions and the query, then pick
# the top-k tools by cosine similarity. Here we simulate with mock embeddings.

import math

# Mock embeddings (in reality, you'd use text-embedding-ada-002 or similar)
MOCK_EMBEDDINGS = {
    "get_weather":     [0.9, 0.1, 0.0, 0.1],
    "calculator":      [0.1, 0.8, 0.0, 0.2],
    "lookup_word":     [0.2, 0.1, 0.9, 0.0],
    "get_current_time":[0.3, 0.2, 0.1, 0.8],
}


def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(x**2 for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


def select_tools_by_embedding(query_embedding: list[float], top_k: int = 2) -> list[str]:
    """Select top-k tools by cosine similarity to the query embedding."""
    scores = [
        (name, cosine_similarity(query_embedding, emb))
        for name, emb in MOCK_EMBEDDINGS.items()
    ]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [name for name, _ in scores[:top_k]]


# Simulate: query about weather -> embedding close to get_weather
query_emb = [0.85, 0.15, 0.05, 0.1]
selected = select_tools_by_embedding(query_emb, top_k=2)
print(f"Top-2 tools for weather-like query: {selected}")

---
## 9. Tool Composition

Complex tasks require **chaining** multiple tools. The output of one tool becomes input to the next. We'll build a `ToolChain` that defines pipelines declaratively.

In [ ]:
class ToolChain:
    """Chain multiple tool calls together, piping results forward."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self._steps: list[tuple[str, Callable]] = []

    def add_step(self, tool_name: str, arg_builder: Callable[[dict], dict]) -> "ToolChain":
        """
        Add a step to the chain.

        Args:
            tool_name: Name of the tool to call.
            arg_builder: Function that takes the previous result (as dict)
                         and returns arguments for this tool.
        """
        self._steps.append((tool_name, arg_builder))
        return self

    def run(self, initial_input: dict | None = None) -> list[dict]:
        """Execute all steps sequentially, returning all intermediate results."""
        results = []
        prev_result = initial_input or {}

        for tool_name, arg_builder in self._steps:
            spec = self.registry.get(tool_name)
            if spec is None:
                results.append({"error": f"Tool '{tool_name}' not found"})
                break

            args = arg_builder(prev_result)
            raw_output = spec.func(**args)
            parsed = json.loads(raw_output) if isinstance(raw_output, str) else raw_output
            results.append({"tool": tool_name, "args": args, "output": parsed})
            prev_result = parsed

        return results


print("ToolChain class defined.")

In [ ]:
# ── Example: chain weather + calculator ──────────────────────────────────────
# "Get the temperature in Tokyo, then convert it to Fahrenheit using the calculator."

chain = ToolChain(registry)

chain.add_step(
    "get_weather",
    lambda prev: {"city": "Tokyo", "units": "celsius"},
)

chain.add_step(
    "calculator",
    lambda prev: {
        # Extract the numeric temperature from the weather result
        "expression": f"{prev['temperature'].replace('C', '')} * 9 / 5 + 32"
    },
)

chain_results = chain.run()

for step in chain_results:
    print(f"Tool: {step['tool']}")
    print(f"  Args  : {step['args']}")
    print(f"  Output: {step['output']}")
    print()

In [ ]:
# ── Parallel tool execution ──────────────────────────────────────────────────
# Sometimes the model requests multiple tools at once (parallel tool calls).
# Our dispatcher already handles this — let's verify.

parallel_calls = [
    {
        "id": "call_p1",
        "type": "function",
        "function": {"name": "get_weather", "arguments": json.dumps({"city": "London"})},
    },
    {
        "id": "call_p2",
        "type": "function",
        "function": {"name": "get_current_time", "arguments": "{}"},
    },
    {
        "id": "call_p3",
        "type": "function",
        "function": {"name": "calculator", "arguments": json.dumps({"expression": "100 / 3"})},
    },
]

parallel_results = dispatcher.dispatch(parallel_calls)

print("Parallel execution results:")
for r in parallel_results:
    print(f"  {r.name}: {r.content} ({r.elapsed_ms}ms)")

---
## 10. Error Handling Patterns

Tools fail. APIs timeout, inputs are malformed, rate limits hit. A robust agent needs **retry**, **fallback**, and **graceful degradation** strategies.

In [ ]:
# ── Pattern 1: Retry with exponential backoff ────────────────────────────────

def retry_tool(
    func: Callable,
    args: dict,
    max_retries: int = 3,
    base_delay: float = 0.5,
) -> dict:
    """Retry a tool call with exponential backoff."""
    for attempt in range(max_retries + 1):
        try:
            result = func(**args)
            return {"success": True, "result": result, "attempts": attempt + 1}
        except Exception as e:
            if attempt == max_retries:
                return {"success": False, "error": str(e), "attempts": attempt + 1}
            delay = base_delay * (2 ** attempt)
            print(f"  Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
            time.sleep(delay)


# Simulate a flaky tool that fails 2 out of 3 times
call_count = 0

def flaky_tool(query: str) -> str:
    global call_count
    call_count += 1
    if call_count % 3 != 0:
        raise ConnectionError("Service temporarily unavailable")
    return json.dumps({"result": f"Answer for '{query}'"})


call_count = 0
result = retry_tool(flaky_tool, {"query": "test"}, max_retries=3, base_delay=0.1)
print(f"Result: {result}")

In [ ]:
# ── Pattern 2: Fallback tools ────────────────────────────────────────────────

def execute_with_fallback(
    primary: tuple[Callable, dict],
    fallbacks: list[tuple[Callable, dict]],
) -> dict:
    """Try the primary tool, then each fallback in order."""
    all_tools = [primary] + fallbacks
    errors = []

    for i, (func, args) in enumerate(all_tools):
        label = "Primary" if i == 0 else f"Fallback {i}"
        try:
            result = func(**args)
            return {"success": True, "source": label, "result": result}
        except Exception as e:
            errors.append(f"{label}: {e}")
            print(f"  {label} failed: {e}")

    return {"success": False, "errors": errors}


# Demo: primary fails, fallback succeeds
def premium_weather(city: str) -> str:
    raise ConnectionError("Premium API is down")

def free_weather(city: str) -> str:
    return json.dumps({"city": city, "temp": "15C", "source": "free-tier"})


result = execute_with_fallback(
    primary=(premium_weather, {"city": "Berlin"}),
    fallbacks=[(free_weather, {"city": "Berlin"})],
)
print(f"\nFinal result: {result}")

In [ ]:
# ── Pattern 3: Graceful degradation ──────────────────────────────────────────
# When all tools fail, return a useful message instead of crashing.

def graceful_dispatch(dispatcher: Dispatcher, tool_calls: list[dict]) -> list[dict]:
    """Dispatch tool calls with graceful degradation on failure."""
    results = dispatcher.dispatch(tool_calls)
    messages = []

    for r in results:
        if r.success:
            messages.append({
                "role": "tool",
                "tool_call_id": r.call_id,
                "content": r.content,
            })
        else:
            # Give the model a helpful error so it can adapt
            messages.append({
                "role": "tool",
                "tool_call_id": r.call_id,
                "content": json.dumps({
                    "error": f"Tool '{r.name}' failed.",
                    "suggestion": "Try rephrasing or using a different approach.",
                    "available_tools": dispatcher.registry.list_tools(),
                }),
            })

    return messages


# Test with a mix of valid and invalid calls
mixed_calls = [
    {"id": "ok1", "type": "function", "function": {"name": "calculator", "arguments": json.dumps({"expression": "2+2"})}},
    {"id": "bad1", "type": "function", "function": {"name": "does_not_exist", "arguments": "{}"}},
]

graceful_results = graceful_dispatch(dispatcher, mixed_calls)
for msg in graceful_results:
    print(json.dumps(msg, indent=2))
    print()

---
## 11. Security Considerations

Giving an LLM the ability to execute code via tools introduces real security risks. This section covers the most important defenses.

In [ ]:
# ── 11.1 Input Validation ────────────────────────────────────────────────────
# Never trust arguments from the model without validation.

import re


def validate_args(args: dict, schema_properties: dict) -> tuple[bool, str]:
    """Validate tool arguments against their schema."""
    for key, prop in schema_properties.items():
        if key not in args:
            if key in schema_properties:  # required check would go here
                continue
        value = args.get(key)
        if value is None:
            continue

        expected_type = prop.get("type")
        if expected_type == "string" and not isinstance(value, str):
            return False, f"'{key}' must be a string, got {type(value).__name__}"
        if expected_type == "integer" and not isinstance(value, int):
            return False, f"'{key}' must be an integer, got {type(value).__name__}"

        # Enum validation
        if "enum" in prop and value not in prop["enum"]:
            return False, f"'{key}' must be one of {prop['enum']}, got '{value}'"

    return True, "Valid"


# Test
weather_props = weather_tool["function"]["parameters"]["properties"]

print(validate_args({"city": "Paris", "units": "celsius"}, weather_props))
print(validate_args({"city": "Paris", "units": "kelvin"}, weather_props))
print(validate_args({"city": 123}, weather_props))

In [ ]:
# ── 11.2 Injection Prevention ────────────────────────────────────────────────
# A model could be tricked (via prompt injection) into passing malicious
# arguments to tools — e.g., shell commands inside a "filename" parameter.

DANGEROUS_PATTERNS = [
    r"[;&|`$]",          # shell metacharacters
    r"\.\.[\/\\]",      # path traversal
    r"<script",           # XSS
    r"(?i)(drop|delete|truncate)\s+table",  # SQL injection
]


def sanitize_string(value: str, field_name: str = "input") -> tuple[bool, str]:
    """Check a string argument for injection patterns."""
    for pattern in DANGEROUS_PATTERNS:
        if re.search(pattern, value):
            return False, f"Blocked: suspicious pattern in '{field_name}': matched {pattern}"
    return True, value


# Test
test_inputs = [
    ("San Francisco", "city"),
    ("Paris; rm -rf /", "city"),
    ("../../etc/passwd", "filename"),
    ("<script>alert('xss')</script>", "query"),
    ("normal query", "query"),
]

for value, field in test_inputs:
    ok, msg = sanitize_string(value, field)
    status = "PASS" if ok else "BLOCK"
    print(f"[{status}] {field}='{value}' -> {msg}")

In [ ]:
# ── 11.3 Rate Limiting ───────────────────────────────────────────────────────
# Prevent runaway tool calls from draining API budgets or causing abuse.

class RateLimiter:
    """Simple per-tool rate limiter using a sliding window."""

    def __init__(self, max_calls: int = 10, window_seconds: float = 60.0):
        self.max_calls = max_calls
        self.window = window_seconds
        self._calls: dict[str, list[float]] = {}  # tool_name -> [timestamps]

    def allow(self, tool_name: str) -> bool:
        """Check if a tool call is allowed under the rate limit."""
        now = time.time()
        if tool_name not in self._calls:
            self._calls[tool_name] = []

        # Remove calls outside the window
        self._calls[tool_name] = [
            t for t in self._calls[tool_name] if now - t < self.window
        ]

        if len(self._calls[tool_name]) >= self.max_calls:
            return False

        self._calls[tool_name].append(now)
        return True


# Demo
limiter = RateLimiter(max_calls=3, window_seconds=10.0)

for i in range(5):
    allowed = limiter.allow("calculator")
    print(f"Call {i+1}: {'Allowed' if allowed else 'RATE LIMITED'}")

In [ ]:
# ── 11.4 Sandboxing Concepts ─────────────────────────────────────────────────
# In production, tools that execute code or access files should run in a sandbox.

# This cell is conceptual — showing the ARCHITECTURE, not a live sandbox.

SANDBOX_ARCHITECTURE = """
Sandboxing Strategies for Tool Execution:

1. PROCESS-LEVEL ISOLATION
   - Run each tool in a subprocess with resource limits (CPU, memory, time)
   - Use subprocess.run(timeout=...) as a basic mechanism
   - More robust: use cgroups or Docker containers

2. FILESYSTEM ISOLATION
   - Mount a read-only filesystem or use chroot/containers
   - Whitelist specific directories the tool can access
   - Never let a tool write to arbitrary paths

3. NETWORK ISOLATION
   - Restrict which domains/IPs a tool can reach
   - Use allowlists for external API calls
   - Proxy outbound requests through a gateway

4. PERMISSION MODEL
   - Tools declare required permissions (read_file, write_file, network, etc.)
   - The agent framework checks permissions before execution
   - Users approve sensitive operations (human-in-the-loop)

5. AUDIT LOGGING
   - Log every tool call with arguments, results, and timestamps
   - Flag anomalous patterns (e.g., sudden spike in file system access)
   - Enable post-incident investigation
"""

print(SANDBOX_ARCHITECTURE)

---
## 12. Exercises

### 12.1 Conceptual Questions

1. **Why does function calling reduce hallucination?** Explain the difference between an LLM generating facts from weights vs. reading tool output.

2. **When should you NOT use tools?** Give two scenarios where tool use adds cost/latency without benefit.

3. **What happens if you send 50 tool schemas to the model?** Discuss the tradeoff between tool availability and context window usage.

In [ ]:
# ── 12.2 Coding Exercise: Build a Calculator Tool Registry ────────────────────
#
# TASK: Build a ToolRegistry with four math tools:
#   - add(a, b)       -> a + b
#   - subtract(a, b)  -> a - b
#   - multiply(a, b)  -> a * b
#   - divide(a, b)    -> a / b  (handle division by zero!)
#
# Then build a Dispatcher and test it with the mock_calls below.

# --- YOUR CODE HERE ---

def add(a: float, b: float) -> str:
    """Add two numbers."""
    pass  # TODO: implement

def subtract(a: float, b: float) -> str:
    """Subtract b from a."""
    pass  # TODO: implement

def multiply(a: float, b: float) -> str:
    """Multiply two numbers."""
    pass  # TODO: implement

def divide(a: float, b: float) -> str:
    """Divide a by b."""
    pass  # TODO: implement


# Create registry and register all four tools
# math_registry = ToolRegistry()
# ...

# Create dispatcher
# math_dispatcher = Dispatcher(math_registry)

# --- END YOUR CODE ---

In [ ]:
# ── Test your calculator registry ────────────────────────────────────────────

calc_test_calls = [
    {"id": "t1", "type": "function", "function": {"name": "add", "arguments": json.dumps({"a": 10, "b": 5})}},
    {"id": "t2", "type": "function", "function": {"name": "subtract", "arguments": json.dumps({"a": 10, "b": 5})}},
    {"id": "t3", "type": "function", "function": {"name": "multiply", "arguments": json.dumps({"a": 10, "b": 5})}},
    {"id": "t4", "type": "function", "function": {"name": "divide", "arguments": json.dumps({"a": 10, "b": 0})}},
]

# Uncomment after implementing:
# results = math_dispatcher.dispatch(calc_test_calls)
# for r in results:
#     print(f"{r.name}: {r.content}")

print("Uncomment the test code above after implementing the exercise.")

In [ ]:
# ── 12.2 Solution ────────────────────────────────────────────────────────────
#
# Uncomment to see the reference implementation.

# def add(a: float, b: float) -> str:
#     return json.dumps({"result": a + b})
#
# def subtract(a: float, b: float) -> str:
#     return json.dumps({"result": a - b})
#
# def multiply(a: float, b: float) -> str:
#     return json.dumps({"result": a * b})
#
# def divide(a: float, b: float) -> str:
#     if b == 0:
#         return json.dumps({"error": "Division by zero"})
#     return json.dumps({"result": a / b})
#
# math_registry = ToolRegistry()
# math_registry.register(add, description="Add two numbers",
#     param_descriptions={"a": "First number", "b": "Second number"}, tags=["math"])
# math_registry.register(subtract, description="Subtract b from a",
#     param_descriptions={"a": "First number", "b": "Second number"}, tags=["math"])
# math_registry.register(multiply, description="Multiply two numbers",
#     param_descriptions={"a": "First number", "b": "Second number"}, tags=["math"])
# math_registry.register(divide, description="Divide a by b",
#     param_descriptions={"a": "Dividend", "b": "Divisor (cannot be zero)"}, tags=["math"])
#
# math_dispatcher = Dispatcher(math_registry)
# results = math_dispatcher.dispatch(calc_test_calls)
# for r in results:
#     print(f"{r.name}: {r.content}")

print("Solution available — uncomment to run.")

### 12.3 Design Exercise: Plugin System

**Scenario:** You are building an agent framework that supports third-party plugins. Each plugin is a Python package that registers one or more tools when imported.

**Design the system:**

1. How should plugins declare their tools? (entry points, decorator, config file?)
2. How do you handle version conflicts between plugins?
3. What security checks should run before a plugin's tools are available to the agent?
4. How do you handle a plugin that takes 30 seconds to respond?

Write your design as a class skeleton below.

In [ ]:
# ── Design skeleton for a Plugin System ──────────────────────────────────────
#
# This is a DESIGN exercise — fill in docstrings and method signatures.
# No need to implement the bodies.

class PluginManifest:
    """Describes a plugin: its tools, required permissions, and version."""
    name: str
    version: str
    tools: list[dict]         # list of tool schemas
    permissions: list[str]    # e.g., ["network", "filesystem:read"]


class PluginManager:
    """Discovers, validates, and loads plugins into a ToolRegistry."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.loaded_plugins: dict[str, PluginManifest] = {}

    def discover(self, plugin_dir: str) -> list[PluginManifest]:
        """Scan a directory for plugin packages. Each must have a manifest.json."""
        # TODO: Your design here
        pass

    def validate(self, manifest: PluginManifest) -> tuple[bool, str]:
        """Check version conflicts, permission safety, schema validity."""
        # TODO: Your design here
        pass

    def load(self, manifest: PluginManifest) -> bool:
        """Import the plugin module and register its tools."""
        # TODO: Your design here
        pass

    def unload(self, plugin_name: str) -> bool:
        """Remove a plugin's tools from the registry."""
        # TODO: Your design here
        pass


print("Plugin system skeleton defined. Fill in the design as an exercise.")

---
## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Hallucination Problem | Tools ground LLM outputs in real data |
| Function Calling Protocol | 3-phase handshake: schema, tool_call, tool result |
| JSON Schema | Describes tool parameters so the model knows what to pass |
| Tool Registry | Central catalog of tools with metadata and tags |
| Dispatcher | Routes model's tool_calls to implementations and collects results |
| Result Parsing | Format tool outputs as messages the model can consume |
| Dynamic Selection | Filter tools by keyword, tag, or embedding similarity |
| Tool Composition | Chain tools together — output of one feeds into the next |
| Error Handling | Retry, fallback, graceful degradation |
| Security | Validate inputs, prevent injection, rate-limit, sandbox |

**Next:** Chapter 7 — Memory Systems